# Gomory-Hu Fraud Detection — Full Pipeline

Runs the complete pipeline end-to-end:
1. Build the transaction graph from the IEEE-CIS dataset (or synthetic fallback)
2. Extract the largest connected component
3. Benchmark our Gusfield implementation vs. NetworkX
4. Score nodes, sweep thresholds, evaluate against ground-truth fraud labels
5. Plot precision vs. recall
6. Generate an interactive Pyvis visualisation

In [ ]:
import sys, os, time
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from src.graph_builder  import build_transaction_graph, get_largest_connected_component
from src.gomory_hu      import gusfield
from src.fraud_signal   import compute_suspicion_scores, flag_suspicious_nodes, evaluate
from src.visualize      import visualize_tree

os.makedirs('../output', exist_ok=True)

## 1. Data Loading

Falls back to a 30-node synthetic graph when the IEEE-CIS CSVs are absent.

In [ ]:
import random

def make_synthetic_graph(n_nodes=30, fraud_rate=0.25, seed=42):
    """Generate a synthetic fraud graph with a dense fraud cluster weakly
    connected to a legitimate cluster via a handful of bridge edges."""
    random.seed(seed)
    nodes = list(range(n_nodes))
    n_fraud = int(n_nodes * fraud_rate)
    fraud_labels = {n: (1 if n < n_fraud else 0) for n in nodes}
    fraud_nodes  = [n for n in nodes if fraud_labels[n] == 1]
    legit_nodes  = [n for n in nodes if fraud_labels[n] == 0]
    graph = {n: {} for n in nodes}

    # Dense fraud cluster — high internal weights.
    for i in range(len(fraud_nodes)):
        for j in range(i + 1, len(fraud_nodes)):
            w = random.randint(5, 10)
            graph[fraud_nodes[i]][fraud_nodes[j]] = w
            graph[fraud_nodes[j]][fraud_nodes[i]] = w

    # Legitimate cluster — moderately connected.
    for i in range(len(legit_nodes)):
        for j in range(i + 1, len(legit_nodes)):
            if random.random() < 0.45:
                w = random.randint(3, 8)
                graph[legit_nodes[i]][legit_nodes[j]] = w
                graph[legit_nodes[j]][legit_nodes[i]] = w

    # Weak bridges: each fraud node connects to one random legit node.
    for f in fraud_nodes:
        l = random.choice(legit_nodes)
        w = random.randint(1, 2)
        graph[f][l] = w
        graph[l][f] = w

    return graph, fraud_labels


TRANSACTION_PATH = '../data/train_transaction.csv'
IDENTITY_PATH    = '../data/train_identity.csv'

if os.path.exists(TRANSACTION_PATH) and os.path.exists(IDENTITY_PATH):
    print('Loading IEEE-CIS dataset...')
    graph_full, fraud_labels = build_transaction_graph(TRANSACTION_PATH, IDENTITY_PATH)
    data_source = 'IEEE-CIS'
else:
    print('IEEE-CIS data not found — using 30-node synthetic graph.')
    graph_full, fraud_labels = make_synthetic_graph(n_nodes=30)
    data_source = 'Synthetic'

print(f'Source: {data_source}')
print(f'Raw graph: {len(graph_full):,} nodes')

## 2. Graph Stats & Largest Connected Component

In [ ]:
def graph_stats(g, labels, label=''):
    n = len(g)
    e = sum(len(v) for v in g.values()) // 2
    n_fraud = sum(1 for v in labels.values() if v == 1)
    deg = [len(v) for v in g.values()]
    print(f"{label}")
    print(f"  Nodes          : {n:,}")
    print(f"  Edges          : {e:,}")
    print(f"  Fraud nodes    : {n_fraud:,}  ({100*n_fraud/n:.1f}%)")
    print(f"  Avg degree     : {sum(deg)/n:.2f}")
    print(f"  Max degree     : {max(deg)}")

graph_stats(graph_full, fraud_labels, 'Full graph')

graph = get_largest_connected_component(graph_full)
# Restrict fraud_labels to LCC nodes.
lcc_labels = {n: fraud_labels[n] for n in graph if n in fraud_labels}

print()
graph_stats(graph, lcc_labels, 'Largest connected component')

## 3. Gusfield vs. NetworkX — Runtime Benchmark

In [ ]:
# --- Our Gusfield implementation ---
t0 = time.perf_counter()
tree, labels = gusfield(graph)
t_ours = time.perf_counter() - t0
print(f'Ours  (Gusfield): {t_ours:.3f}s   |  tree edges: {len(tree)}')

# --- NetworkX gomory_hu_tree ---
G_nx = nx.Graph()
for u, nbrs in graph.items():
    for v, w in nbrs.items():
        if not G_nx.has_edge(u, v):
            G_nx.add_edge(u, v, capacity=w)

t0 = time.perf_counter()
T_nx = nx.gomory_hu_tree(G_nx, capacity='capacity')
t_nx = time.perf_counter() - t0
print(f'NetworkX        : {t_nx:.3f}s   |  tree edges: {T_nx.number_of_edges()}')

In [ ]:
# Runtime comparison table.
ratio = t_nx / t_ours if t_ours > 0 else float('inf')
comparison = pd.DataFrame({
    'Implementation': ['Ours (Gusfield from scratch)', 'NetworkX gomory_hu_tree'],
    'Time (s)':       [round(t_ours, 4), round(t_nx, 4)],
    'Relative':       [f'1.0×', f'{ratio:.2f}×'],
})
print(comparison.to_string(index=False))

## 4. Suspicion Scoring

In [ ]:
scores = compute_suspicion_scores(tree, labels)

score_vals = list(scores.values())
print(f'Nodes scored: {len(scores):,}')
print(f'Score range : {min(score_vals):.2f} – {max(score_vals):.2f}')
print(f'Score mean  : {sum(score_vals)/len(score_vals):.2f}')

plt.figure(figsize=(8, 3))
plt.hist(score_vals, bins=40, color='steelblue', edgecolor='white')
plt.xlabel('Suspicion score (min incident tree-edge weight)')
plt.ylabel('Node count')
plt.title('Distribution of suspicion scores')
plt.tight_layout()
plt.savefig('../output/score_distribution.png', dpi=120)
plt.show()

## 5. Threshold Sweep — Evaluation Table

In [ ]:
thresholds = [5.0, 10.0, 15.0, 20.0]
rows = []

for pct in thresholds:
    flagged = flag_suspicious_nodes(scores, percentile_threshold=pct)
    metrics = evaluate(flagged, lcc_labels)
    rows.append({
        'Threshold (%)':   pct,
        'Flagged':         metrics['total_flagged'],
        'TP':              metrics['true_positives'],
        'FP':              metrics['false_positives'],
        'FN':              metrics['false_negatives'],
        'Precision':       metrics['precision'],
        'Recall':          metrics['recall'],
        'F1':              metrics['f1'],
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

## 6. Precision vs. Recall Curve

In [ ]:
# Sweep finer percentile grid to draw the full curve.
fine_thresholds = list(range(1, 51))  # 1% to 50%
precisions, recalls, f1s = [], [], []

for pct in fine_thresholds:
    flagged = flag_suspicious_nodes(scores, percentile_threshold=pct)
    m = evaluate(flagged, lcc_labels)
    precisions.append(m['precision'])
    recalls.append(m['recall'])
    f1s.append(m['f1'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(recalls, precisions, marker='.', color='steelblue')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision vs. Recall')
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

axes[1].plot(fine_thresholds, f1s, color='darkorange')
axes[1].set_xlabel('Percentile threshold (%)')
axes[1].set_ylabel('F1 score')
axes[1].set_title('F1 score vs. flag threshold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../output/precision_recall.png', dpi=120)
plt.show()

## 7. Interactive Gomory-Hu Tree Visualisation

In [ ]:
output_html = '../output/fraud_tree.html'
visualize_tree(tree, labels, scores, lcc_labels, output_path=output_html)
print(f'Saved to {output_html}')
print('Open the file in a browser to explore the interactive tree.')

## 8. Interpretation

The Gomory-Hu tree exposes the global min-cut structure of the transaction
graph in a single spanning tree, making fraud rings visible as weakly-attached
subtrees: a ring of cards that share device fingerprints and billing addresses
internally but connect to the honest population via only a few transactions
will have a very low min-cut edge anchoring it to the tree.
The suspicion score (minimum incident tree-edge weight) therefore works best
when fraudsters form a dense cluster with sparse external links — the classic
card-testing or account-takeover ring pattern.
Where it struggles is against solo fraudsters who mimic legitimate spending
patterns; such nodes are structurally indistinguishable from honest high-volume
cards, so their tree edges carry normal weights and the score does not flag them.
A natural next step is to combine this structural signal with transaction-level
features (amount, time-of-day, velocity) to build a hybrid ranker.